# Deep Custom CNN for Traffic Density Classification

Notebook này khảo sát Deep CNN huấn luyện từ đầu để phân loại ảnh thành năm mức mật độ giao thông. Các thí nghiệm giữ nguyên kiến trúc và pipeline dữ liệu, đồng thời so sánh ảnh hưởng riêng của augmentation và class weights trên validation.


In [ ]:
import ctypes
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_project_dir():
    """Find this repository even when VS Code starts Jupyter from $HOME."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates.append(Path.home() / "Documents" / "tempt" / "dl-traffic-density-classi")
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / ".venv").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Không tìm thấy project directory chứa src/ và .venv/. "
        f"Jupyter current directory: {Path.cwd()}"
    )


project_dir = find_project_dir()

# VS Code may start an ipykernel without LD_LIBRARY_PATH. Preload CUDA shared
# libraries from THIS repository's .venv before importing TensorFlow.
CUDA_LIBRARY_ROOT = (
    project_dir / ".venv" / "lib" / "python3.11" / "site-packages" / "nvidia"
)
loaded_cuda_libraries = set()
for _ in range(4):
    for library in sorted(CUDA_LIBRARY_ROOT.glob("*/lib/*.so*")):
        if library in loaded_cuda_libraries:
            continue
        try:
            ctypes.CDLL(str(library), mode=ctypes.RTLD_GLOBAL)
            loaded_cuda_libraries.add(library)
        except OSError:
            # Optional NVSHMEM/MPI libraries are not needed by TensorFlow.
            pass

import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models

GPUS = tf.config.list_physical_devices("GPU")
if not GPUS:
    raise RuntimeError(
        "TensorFlow vẫn không thấy GPU sau CUDA preload. "
        "Restart Kernel, sau đó chạy riêng cell này trước mọi cell TensorFlow khác."
    )
try:
    for gpu in GPUS:
        tf.config.experimental.set_memory_growth(gpu, True)
except RuntimeError as error:
    print(f"Memory growth was already configured: {error}")
tf.keras.mixed_precision.set_global_policy("mixed_float16")

print("Project directory:", project_dir)
print(f"Preloaded CUDA libraries: {len(loaded_cuda_libraries)}")
print("GPUs:", GPUS)
print("Mixed precision:", tf.keras.mixed_precision.global_policy().name)


In [ ]:
# project_dir is resolved in the GPU bootstrap cell. This fallback only helps
# when this cell is run independently after a kernel restart.
if "project_dir" not in globals() or not (project_dir / "src").exists():
    project_dir = Path.home() / "Documents" / "tempt" / "dl-traffic-density-classi"
if not (project_dir / "src").exists():
    raise FileNotFoundError(f"Cannot find src/ under project directory: {project_dir}")
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

from src.data_loader import CLASS_NAMES, create_datasets
from src.evaluation import evaluate_model


## 1. Thiết lập thí nghiệm

In [ ]:
SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 64
EPOCHS = 20

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

if not GPUS:
    raise RuntimeError("GPU check failed in the first notebook cell.")

RUN_DIR = project_dir / "outputs" / "complex_cnn_notebook"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", EPOCHS)
print("Outputs:", RUN_DIR)
print("GPUs:", GPUS)
print("Mixed precision:", tf.keras.mixed_precision.global_policy().name)


Dùng cùng seed, kích thước ảnh và batch size như pipeline chung. Cùng một cấu hình callbacks ?ược dùng cho cả ba experiment. EarlyStopping theo dõi `val_loss`, bắt ?ầu xét dừng sau epoch 5 ?ể mô hình có thời gian warm-up; `patience=6` giúp tránh dừng vì dao ?ộng validation ngắn hạn. ReduceLROnPlateau giảm learning rate khi `val_loss` không cải thiện trong 3 epoch. Thiết lập này không ?ảm bảo accuracy sẽ tăng.


## 2. Đọc dữ liệu

In [ ]:
train_dataset, val_dataset, test_dataset = create_datasets(
    augmentation=False,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

print("Classes:", CLASS_NAMES)
print("Training batches:", train_dataset.cardinality().numpy())
print("Validation batches:", val_dataset.cardinality().numpy())
print("Test batches:", test_dataset.cardinality().numpy())


Dữ liệu ?ược ?ọc từ các manifest ?ã làm sạch bằng pipeline tf.data.Dataset. Augmentation chỉ áp dụng cho tập train ở những thí nghiệm có bật augmentation; validation và test ?ược giữ nguyên ?ể ?ánh giá công bằng.


## 3. Phân bố lớp và class weights

In [ ]:
train_manifest = pd.read_csv(project_dir / "data" / "manifests" / "train.csv")
class_to_index = {name: index for index, name in enumerate(CLASS_NAMES)}
train_counts = train_manifest["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0)
train_class_ids = train_manifest["class_name"].map(class_to_index).to_numpy()
class_ids = np.arange(len(CLASS_NAMES))

class_weight_values = compute_class_weight(
    class_weight="balanced", classes=class_ids, y=train_class_ids,
)
class_weights = {
    int(class_id): float(weight)
    for class_id, weight in zip(class_ids, class_weight_values)
}
class_weight_table = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Number of images": train_counts.to_numpy(),
    "Class weight": [class_weights[index] for index in class_ids],
})
class_weight_table

Phân bố lớp cho thấy dữ liệu bị mất cân bằng, nên Experiment 3 thử class weights để tăng ảnh hưởng của các lớp ít mẫu trong loss. Trọng số chỉ được tính từ nhãn tập train và chỉ truyền vào quá trình huấn luyện; validation và test không dùng class weights.

## 4. Kiến trúc Deep Custom CNN

In [ ]:
def build_deep_cnn():
    model = models.Sequential([
        layers.Input(shape=(*IMAGE_SIZE, 3)),
        layers.Conv2D(32, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.Conv2D(32, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.Conv2D(64, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.Conv2D(128, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(256, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(len(CLASS_NAMES), activation="softmax", dtype="float32"),
    ])
    return model

model_preview = build_deep_cnn()
model_preview.summary()
M1_PARAMETERS = 2_792_869
print(f"M1 parameters: {M1_PARAMETERS:,}")
print(f"M2 parameters: {model_preview.count_params():,}")

So với Simple CNN ở M1, Deep CNN dùng nhiều convolution blocks hơn để học đặc trưng theo từng mức. BatchNormalization hỗ trợ ổn định quá trình học; GlobalAveragePooling thay Flatten để giảm số tham số ở phần phân loại; Dropout hỗ trợ hạn chế overfitting. Mục tiêu là cải thiện feature extraction và khả năng tổng quát hóa.

## 5. Hàm hỗ trợ huấn luyện và trực quan hóa

In [ ]:
class ValidationMacroF1(tf.keras.callbacks.Callback):
    def __init__(self, dataset):
        super().__init__()
        self.dataset = dataset

    def on_epoch_end(self, epoch, logs=None):
        probabilities = self.model.predict(self.dataset, verbose=0)
        y_pred = np.argmax(probabilities, axis=1)
        y_true = get_true_labels(self.dataset)
        logs["val_macro_f1"] = f1_score(
            y_true, y_pred, average="macro", zero_division=0,
        )


def make_callbacks(val_dataset, checkpoint_path):
    # Selection, checkpointing and early stopping all use the same validation metric.
    return [
        ValidationMacroF1(val_dataset),
        tf.keras.callbacks.ModelCheckpoint(
            checkpoint_path,
            monitor="val_macro_f1",
            mode="max",
            save_best_only=True,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_macro_f1",
            mode="max",
            patience=6,
            restore_best_weights=True,
            start_from_epoch=5,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_macro_f1",
            mode="max",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1,
        ),
    ]


def plot_history(history, title):
    epochs = range(1, len(history.history["loss"]) + 1)
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history.history["loss"], label="Training loss")
    plt.plot(epochs, history.history["val_loss"], label="Validation loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title(f"{title} - Loss"); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history.history["accuracy"], label="Training accuracy")
    plt.plot(epochs, history.history["val_accuracy"], label="Validation accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title(f"{title} - Accuracy"); plt.legend()
    plt.tight_layout()
    plt.show()


def get_true_labels(dataset):
    return np.concatenate([np.argmax(labels.numpy(), axis=1) for _, labels in dataset])


def get_validation_macro_f1(model, dataset):
    probabilities = model.predict(dataset, verbose=0)
    predicted_labels = np.argmax(probabilities, axis=1)
    return f1_score(
        get_true_labels(dataset), predicted_labels, average="macro", zero_division=0,
    )


def record_experiment(name, history, model, val_dataset, augmentation, weighted):
    experiment_models[name] = model
    experiment_histories[name] = history
    experiment_records.append({
        "Experiment": name,
        "Augmentation": augmentation,
        "Class Weights": weighted,
        "Best Val Accuracy": max(history.history["val_accuracy"]),
        "Best Val Macro F1": get_validation_macro_f1(model, val_dataset),
        "Lowest Val Loss": min(history.history["val_loss"]),
        "Epochs Trained": len(history.history["loss"]),
    })
    result = experiment_records[-1]
    print(f"Best validation accuracy: {result['Best Val Accuracy']:.4f}")
    print(f"Validation macro F1: {result['Best Val Macro F1']:.4f}")
    print(f"Lowest validation loss: {result['Lowest Val Loss']:.4f}")


Callbacks theo dõi `val_loss`. EarlyStopping dùng `start_from_epoch=5` ?ể mô hình warm-up trước khi xét dừng, cùng `patience=6` ?ể chịu ?ược dao ?ộng validation ngắn hạn. ReduceLROnPlateau giảm learning rate khi validation loss không cải thiện trong 3 epoch. Cùng `make_callbacks()` ?ược dùng cho cả ba experiment ?ể so sánh công bằng. Thiết lập này không ?ảm bảo accuracy sẽ tăng.


## Visual inspection of augmentation

Trước khi huấn luyện, hiển thị các ví dụ ảnh không augmentation và có augmentation từ cùng train split. Hai hàng ?ược lấy từ các dataset ?ược shuffle riêng, nên các ảnh cùng cột không nhất thiết là cùng sample.


In [ ]:
original_train_dataset, _, _ = create_datasets(
    augmentation=False,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)
augmented_train_dataset, _, _ = create_datasets(
    augmentation=True,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

original_images, original_labels = next(iter(original_train_dataset))
augmented_images, augmented_labels = next(iter(augmented_train_dataset))
num_images = min(4, int(original_images.shape[0]), int(augmented_images.shape[0]))
fig, axes = plt.subplots(2, num_images, figsize=(4 * num_images, 7), squeeze=False)

for index in range(num_images):
    original_class = CLASS_NAMES[np.argmax(original_labels[index].numpy())]
    augmented_class = CLASS_NAMES[np.argmax(augmented_labels[index].numpy())]

    axes[0, index].imshow(original_images[index].numpy())
    axes[0, index].set_title(f"Without augmentation\n{original_class}")
    axes[0, index].axis("off")

    axes[1, index].imshow(augmented_images[index].numpy())
    axes[1, index].set_title(f"With augmentation\n{augmented_class}")
    axes[1, index].axis("off")

plt.tight_layout()
plt.show()


## Dataset integrity check

Kiểm tra chỉ-đọc từng ảnh theo manifest. Loader hỗ trợ cả thư mục `Final Dataset/`
đã giải nén và fallback sang `traffic.zip`; không thay đổi split, nhãn hay file ảnh.


In [ ]:
from src.data_loader import read_image

manifest_paths = {
    "TRAIN": project_dir / "data" / "manifests" / "train.csv",
    "VAL": project_dir / "data" / "manifests" / "val.csv",
    "TEST": project_dir / "data" / "manifests" / "test.csv",
}


def find_bad_images(csv_path):
    manifest = pd.read_csv(csv_path)
    bad_images = []
    for relative_path in manifest["image_path"]:
        try:
            read_image(str(project_dir / relative_path))
        except Exception as error:
            bad_images.append({"relative_path": str(relative_path), "error": str(error)})
    return bad_images


bad_images_by_split = {
    split: find_bad_images(csv_path) for split, csv_path in manifest_paths.items()
}
for split, bad_images in bad_images_by_split.items():
    print(f"{split}: {len(bad_images)} bad image(s)")
    for bad_image in bad_images[:20]:
        print(f"  {bad_image['relative_path']}: {bad_image['error']}")
    if len(bad_images) > 20:
        print(f"  ... {len(bad_images) - 20} more")


### Integrity result

Notebook này **không tự sửa hoặc ghi đè ảnh nguồn**. Nếu có ảnh lỗi, ghi lại
đường dẫn để review riêng thay vì sửa JPEG trong khi huấn luyện.


In [ ]:
bad_count = sum(len(images) for images in bad_images_by_split.values())
if bad_count:
    print(f"WARNING: {bad_count} unreadable image(s); stop and review them before training.")
else:
    print("Integrity check passed. No image was modified.")


### Re-check

Cell này chỉ hiển thị lại kết quả kiểm tra hiện tại; không có bước sửa ảnh tự động.


In [ ]:
for split, bad_images in bad_images_by_split.items():
    print(f"{split}: {len(bad_images)} bad image(s)")


### Check available disk space


In [ ]:
import shutil

total, used, free = shutil.disk_usage(project_dir)
print(f"Total: {total / 1024**3:.2f} GB")
print(f"Used : {used / 1024**3:.2f} GB")
print(f"Free : {free / 1024**3:.2f} GB")
if free < 2 * 1024**3:
    print("WARNING: Low disk space. Free some disk space before training.")


In [ ]:
# Initialize tracking after a runtime reset, while preserving an active Experiment 1 record/model.
if "experiment_models" not in globals():
    experiment_models = {}
if "experiment_histories" not in globals():
    experiment_histories = {}
if "experiment_records" not in globals():
    experiment_records = []


## 6. Experiment 1 — Deep CNN baseline

Checkpoint đã có được tải ở phần đánh giá. Cell train bên dưới được comment để `Run All` không huấn luyện lại.


In [ ]:
# TRAINING DISABLED — Experiment 1: Deep CNN baseline
# Run All loads existing checkpoints and evaluates them; it does not retrain.
# To train this experiment again, uncomment this cell intentionally.
#
# Original training code:
# experiment_models = {}
# experiment_histories = {}
# experiment_records = []
#
# train_dataset_1, val_dataset_1, _ = create_datasets(
#     augmentation=False, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
# )
# model_exp1 = build_deep_cnn()
# model_exp1.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
#     loss="categorical_crossentropy", metrics=["accuracy"], steps_per_execution=16,
# )
# history_exp1 = model_exp1.fit(
#     train_dataset_1, validation_data=val_dataset_1, epochs=EPOCHS,
#     callbacks=make_callbacks(val_dataset_1, RUN_DIR / "exp1_best.keras"),
# )
# plot_history(history_exp1, "Experiment 1: Deep CNN baseline")
# record_experiment(
#     "Deep CNN baseline", history_exp1, model_exp1, val_dataset_1,
#     augmentation=False, weighted=False,
# )


## 7. Experiment 2 — Deep CNN + Augmentation

Checkpoint đã có được tải ở phần đánh giá. Cell train bên dưới được comment để `Run All` không huấn luyện lại.


In [ ]:
# TRAINING DISABLED — Experiment 2: Deep CNN + Augmentation
# Run All loads existing checkpoints and evaluates them; it does not retrain.
# To train this experiment again, uncomment this cell intentionally.
#
# Original training code:
# train_dataset_2, val_dataset_2, _ = create_datasets(
#     augmentation=True, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
# )
# model_exp2 = build_deep_cnn()
# model_exp2.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
#     loss="categorical_crossentropy", metrics=["accuracy"], steps_per_execution=16,
# )
# history_exp2 = model_exp2.fit(
#     train_dataset_2, validation_data=val_dataset_2, epochs=EPOCHS,
#     callbacks=make_callbacks(val_dataset_2, RUN_DIR / "exp2_best.keras"),
# )
# plot_history(history_exp2, "Experiment 2: Deep CNN + Augmentation")
# record_experiment(
#     "Deep CNN + Augmentation", history_exp2, model_exp2, val_dataset_2,
#     augmentation=True, weighted=False,
# )


In [ ]:
from PIL import Image
import pandas as pd

train_manifest = pd.read_csv(project_dir / "data" / "manifests" / "train.csv")
bad_images = []

for rel_path in train_manifest["image_path"]:
    path = project_dir / rel_path
    try:
        with Image.open(path) as image:
            image.load()
    except Exception as e:
        bad_images.append((rel_path, str(e)))

bad_images

In [ ]:
import src.data_loader as dl

print(dl.__file__)
print(dl.PROJECT_DIR)
print(dl.MANIFEST_DIR)

train_manifest = pd.read_csv(dl.MANIFEST_DIR / "train.csv")
print(train_manifest["image_path"].isin([
    "Final Dataset/training/Traffic Jam/getty_523821065_98040.jpg",
    "Final Dataset/training/Traffic Jam/dreamstime_s_21795079.jpg",
]).to_list())

Kết quả validation được tính lại từ checkpoint khi chạy notebook. Không có huấn luyện lại trong chế độ này.


## 8. Experiment 3 — Deep CNN + Augmentation + Class Weights

Checkpoint đã có được tải ở phần đánh giá. Cell train bên dưới được comment để `Run All` không huấn luyện lại.


In [ ]:
# TRAINING DISABLED — Experiment 3: Deep CNN + Augmentation + Class Weights
# Run All loads existing checkpoints and evaluates them; it does not retrain.
# To train this experiment again, uncomment this cell intentionally.
#
# Original training code:
# train_dataset_3, val_dataset_3, _ = create_datasets(
#     augmentation=True, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
# )
# model_exp3 = build_deep_cnn()
# model_exp3.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
#     loss="categorical_crossentropy", metrics=["accuracy"], steps_per_execution=16,
# )
# history_exp3 = model_exp3.fit(
#     train_dataset_3, validation_data=val_dataset_3, epochs=EPOCHS,
#     callbacks=make_callbacks(val_dataset_3, RUN_DIR / "exp3_best.keras"), class_weight=class_weights,
# )
# plot_history(history_exp3, "Experiment 3: Deep CNN + Augmentation + Class Weights")
# record_experiment(
#     "Deep CNN + Augmentation + Class Weights", history_exp3, model_exp3, val_dataset_3,
#     augmentation=True, weighted=True,
# )


Kết quả validation được tính lại từ checkpoint khi chạy notebook. Không có huấn luyện lại trong chế độ này.


## 9. So sánh các thí nghiệm và chọn cấu hình cuối

In [ ]:
# Evaluation-only mode: load all completed experiment checkpoints, then
# recompute validation metrics on the current validation dataset. No training runs.
CHECKPOINTS = {
    "Deep CNN baseline": {
        "path": RUN_DIR / "exp1_best.keras",
        "augmentation": False,
        "weighted": False,
    },
    "Deep CNN + Augmentation": {
        "path": RUN_DIR / "exp2_best.keras",
        "augmentation": True,
        "weighted": False,
    },
    "Deep CNN + Augmentation + Class Weights": {
        "path": RUN_DIR / "exp3_best.keras",
        "augmentation": True,
        "weighted": True,
    },
}

missing_checkpoints = [
    str(details["path"])
    for details in CHECKPOINTS.values()
    if not details["path"].exists()
]
if missing_checkpoints:
    raise FileNotFoundError(
        "Missing Complex CNN checkpoint(s):\n" + "\n".join(missing_checkpoints)
    )

experiment_models = {}
experiment_histories = {}
experiment_records = []
y_val = get_true_labels(val_dataset)
y_val_one_hot = tf.one_hot(y_val, depth=len(CLASS_NAMES))

for experiment_name, details in CHECKPOINTS.items():
    loaded_model = tf.keras.models.load_model(details["path"], compile=False)
    probabilities = loaded_model.predict(val_dataset, verbose=0)
    predicted_labels = np.argmax(probabilities, axis=1)
    validation_loss = float(tf.reduce_mean(
        tf.keras.losses.categorical_crossentropy(y_val_one_hot, probabilities)
    ))

    experiment_models[experiment_name] = loaded_model
    experiment_records.append({
        "Experiment": experiment_name,
        "Augmentation": details["augmentation"],
        "Class Weights": details["weighted"],
        "Best Val Accuracy": float(np.mean(predicted_labels == y_val)),
        "Best Val Macro F1": float(f1_score(
            y_val, predicted_labels, average="macro", zero_division=0,
        )),
        "Lowest Val Loss": validation_loss,
        "Epochs Trained": "loaded checkpoint",
        "Checkpoint": str(details["path"].relative_to(project_dir)),
    })

print(f"Loaded {len(experiment_records)} checkpoints; no training was run.")


In [ ]:
experiment_results = pd.DataFrame(experiment_records)
experiment_results = experiment_results.sort_values(
    by=["Best Val Macro F1", "Best Val Accuracy", "Lowest Val Loss"],
    ascending=[False, False, True],
    ignore_index=True,
)
experiment_results


In [ ]:
selected_experiment = max(
    experiment_records,
    key=lambda row: (
        row["Best Val Macro F1"], row["Best Val Accuracy"], -row["Lowest Val Loss"]
    ),
)
final_model = experiment_models[selected_experiment["Experiment"]]
print("Selected final configuration:", selected_experiment["Experiment"])
print("Selection uses validation Macro F1, then accuracy and loss as tie-breakers.")


Lựa chọn model cuối dựa trên Validation Macro F1. Nếu Macro F1 bằng nhau, Validation Accuracy rồi Lowest Validation Loss ?ược dùng làm tie-breaker. Test set chỉ ?ược dùng sau khi ?ã chốt model.


## 10. Đánh giá mô hình M2 trên test set

In [ ]:
m2_result = evaluate_model(
    model=final_model,
    test_dataset=test_dataset,
    model_name="Deep Custom CNN",
)
m2_result


## 11. Confusion matrix v? classification report

Classification report ?ược dùng ?ể xem precision, recall và F1-score của từng mức mật ?ộ giao thông, ?ặc biệt hữu ích vì dataset có mất cân bằng lớp.


Confusion matrix cho biết số mẫu thật của từng lớp được dự đoán vào mỗi lớp; classification report tóm tắt Precision, Recall và F1 trên test set. Có thể dùng hai kết quả này để xem nhóm lớp mật độ trung gian nào còn dễ nhầm lẫn.

In [ ]:
if final_model is not None:
    pred_probs = final_model.predict(test_dataset, verbose=0)
    y_pred = np.argmax(pred_probs, axis=1)
    y_true = get_true_labels(test_dataset)

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASS_NAMES)))
    plt.figure(figsize=(7, 6))
    plt.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.title("Confusion Matrix - Deep Custom CNN")
    plt.colorbar()
    tick_positions = np.arange(len(CLASS_NAMES))
    plt.xticks(tick_positions, CLASS_NAMES, rotation=35, ha="right")
    plt.yticks(tick_positions, CLASS_NAMES)
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    threshold = cm.max() / 2 if cm.size else 0
    for row in range(cm.shape[0]):
        for column in range(cm.shape[1]):
            plt.text(column, row, format(cm[row, column], "d"),
                     ha="center", va="center",
                     color="white" if cm[row, column] > threshold else "black")
    plt.tight_layout()
    plt.show()

    print(classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    ))
else:
    print("Confusion matrix and classification report skipped because the selected model is unavailable.")


**Metrics theo từng lớp trên test set**

Classification report bên trên hiển precision, recall và F1-score cho từng lớp. Dùng các giá trị mới sau khi chạy notebook ?ể xem lớp nào còn khó phân loại.


## 12. So sánh với Simple CNN của M1

C?c gi? tr? M1 b?n d??i l? k?t qu? test ?? ch?t c?a Simple CNN. N?u M1 ???c ch?y l?i v? k?t qu? thay ??i, c?n c?p nh?t c?c gi? tr? n?y tr??c khi so s?nh.


In [ ]:
m1_result = None
print("M1 metrics are not hard-coded here. Load an actual M1 result before comparison.")
if final_model is not None and "m2_result" in globals():
    print("M2 result:")
    display(m2_result)


In [ ]:
if final_model is not None and "m2_result" in globals():
    m1_accuracy = float(m1_result.loc[0, "Accuracy"])
    m1_macro_f1 = float(m1_result.loc[0, "F1-score"])
    m2_accuracy = float(m2_result.loc[0, "Accuracy"])
    m2_macro_f1 = float(m2_result.loc[0, "F1-score"])

    print(f"Accuracy improved over M1: {m2_accuracy > m1_accuracy}")
    print(f"Macro F1 improved over M1: {m2_macro_f1 > m1_macro_f1}")
    print(f"Accuracy difference (M2 - M1): {m2_accuracy - m1_accuracy:+.4f}")
    print(f"Macro F1 difference (M2 - M1): {m2_macro_f1 - m1_macro_f1:+.4f}")
else:
    print("M1/M2 test comparison skipped because the selected model is unavailable.")


Bảng validation và kết quả test ?ược tạo tự ?ộng khi chạy lại notebook. So sánh M2 với M1 cần dựa trên kết quả mới sau khi ba experiment hoàn tất; không diễn giải các output ?ã lưu từ lần chạy trước như kết quả của lần chạy mới.


## 13. Nhận xét

Trong ba experiment, c?u h?nh ???c ch?n theo Validation Macro F1, sau ?? d?ng Validation Accuracy v? Lowest Validation Loss ?? ph? h?a. Experiment 1 c? th? ???c kh?i ph?c v?o b?ng t? metrics ?? l?u d? model kh?ng c?n trong RAM. N?u Experiment 1 th?ng nh?ng model kh?ng c? s?n, c?c b??c ??nh gi? test, confusion matrix, classification report v? l?u model s? ???c b? qua an to?n. Kh?ng suy r?ng k?t qu? c?a c?u h?nh ?? th? th?nh quy t?c chung.


## 14. Lưu mô hình

In [ ]:
if final_model is not None:
    model_path = RUN_DIR / "selected_deep_custom_cnn.keras"
    final_model.save(model_path)
    print(f"Model saved to: {model_path}")


## Kết luận M2

Sau khi chạy lại notebook, tổng hợp kết quả ba experiment từ bảng validation và báo cáo metrics, confusion matrix cùng classification report của model ?ược chọn. Các kết luận về augmentation, class weights và so sánh với M1 cần dựa trên output mới.
